# Features de Agregación por Ventanas de Tiempo (Velocity Features)

En este notebook, se analizan fetaures de agragación basadas en ventanas de tiempo, esto con el fin de analizar si este tipo de features puede contener coimportamientos que se reflejen con el fraude, para ello hay que considerar lo siguiente:

- No existe ninguna columna tipo `user_id`, `payer_id` o similar, es decir, aunque el datset regsitra transacciones, no hay un ID que identifique propiamente a un usuario.
- La única columna de la cual se podría calcular una agregación es la variable `j`, su alta cardinalidad (~7,558-8,324 valores, ~12-13 transacciones cada una en promedio) podría asumirse como un identificador que eprmita calcular fetaures de agragación para esos ID, a diferencia de la columna `k` que aunque es un indicativo este es único para cada registro lo cual no sería candidata para construir este tipo de features.

**Hipótesis general:**
- Construir fetaures de agregación respecto a la columna `j`

**Agregaciones disponibles:**

- `sum`
- `mean`
- `median`
- `max`
- `min`
- `monto`

Se calculan sobre 6 ventanas de tiempo **(1h, 1d, 3d, 7d, 10d, 15d)**, calculadas de forma **causal**: cada fila solo agrega transacciones *anteriores* a su propio timestamp (no incluye la fila actual ni el futuro), para evitar fuga de información.

Al final se repite el mismo análisis estadístico feature-vs-target del notebook anterior (`01_statistical_analysis.ipynb`), mismos umbrales de interpretación, aplicado únicamente a estas features nuevas, para concluir si aportan poder predictivo real.

# Imports

In [1]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd().parent if (Path.cwd().name == "02_feature_engineer") else Path.cwd()
sys.path.insert(0, str(ROOT_DIR))

from src.feature_engineer_utils.aggregation_features import AggregationFeatures, WINDOW_ALIASES
from src.feature_engineer_utils.statistical_tests import FeatureStatTests

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


# Configs

In [2]:
DATASET_DIR = ROOT_DIR / "dataset"
TRAIN_PATH = DATASET_DIR / "train.csv"
OUTPUT_TRAIN_PATH = DATASET_DIR / "train_agg_features.csv"

STATS_OUTPUT_DIR = ROOT_DIR / "02_feature_engineer" / "agg_stats_outputs"
STATS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AGG_STATS_PATH = STATS_OUTPUT_DIR / "stats_fraud_agg_features.csv"

TARGET_COL = "fraude"
ENTITY_COL = "j"
DATE_COL = "fecha"
VALUE_COL = "monto"
WINDOWS = list(WINDOW_ALIASES)  # ["1h", "1d", "3d", "7d", "10d", "15d"]
AGGREGATIONS = ["sum_window", "mean_window", "median_window", "max_window", "min_window"]

TRAIN_PATH


PosixPath('/Users/dpiedrahita/proyectos/DS_pro/dataset/train.csv')

# Read Data

In [3]:
df = pd.read_csv(TRAIN_PATH, parse_dates=["fecha"])
print(f"Shape train: {df.shape}")
df.head()


Shape train: (104981, 19)


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude
0,4,0.7388,6314.50,14.0,0.139279,24.0,BR,7,cat_381751d,0.937548,2361.0,442.0,1,NaN,Y,2020-03-08 00:02:15,22.18,25,0
1,4,0.7548,21171.09,20.0,0.514815,7.0,BR,2,cat_a024847,0.791998,2324.0,73.0,1,NaN,N,2020-03-08 00:04:25,6.00,7,0
2,4,0.9026,4012.83,50.0,0.274167,1.0,BR,3,cat_1d61c62,0.688592,235.0,232.0,1,N,Y,2020-03-08 00:08:23,26.67,91,1
3,4,0.8285,99612.95,1.0,0.000000,4.0,BR,28,cat_01a1725,0.654161,658.0,0.0,1,N,N,2020-03-08 00:08:39,40.01,91,1
4,2,0.5992,53526.36,2.0,0.000000,264.0,AR,34,cat_f1e7464,0.532994,2400.0,10.0,1,NaN,N,2020-03-08 00:09:01,6.25,93,0


## Construcción de las features de agregación

Se generan las 25 combinaciones (5 agregaciones × 5 ventanas) de `monto` agrupado por `j`, todas causales (`closed="left"`, sin incluir la fila actual ni el futuro).

In [4]:
config_list = [
    {
        "process": process,
        "column": VALUE_COL,
        "new_column": f"{VALUE_COL}_{process.replace('_window', '')}_{window}",
        "params": {"window": window},
    }
    for window in WINDOWS
    for process in AGGREGATIONS
]

print(f"Total de features a construir: {len(config_list)}")
config_list


Total de features a construir: 30


[{'process': 'sum_window',
  'column': 'monto',
  'new_column': 'monto_sum_1h',
  'params': {'window': '1h'}},
 {'process': 'mean_window',
  'column': 'monto',
  'new_column': 'monto_mean_1h',
  'params': {'window': '1h'}},
 {'process': 'median_window',
  'column': 'monto',
  'new_column': 'monto_median_1h',
  'params': {'window': '1h'}},
 {'process': 'max_window',
  'column': 'monto',
  'new_column': 'monto_max_1h',
  'params': {'window': '1h'}},
 {'process': 'min_window',
  'column': 'monto',
  'new_column': 'monto_min_1h',
  'params': {'window': '1h'}},
 {'process': 'sum_window',
  'column': 'monto',
  'new_column': 'monto_sum_1d',
  'params': {'window': '1d'}},
 {'process': 'mean_window',
  'column': 'monto',
  'new_column': 'monto_mean_1d',
  'params': {'window': '1d'}},
 {'process': 'median_window',
  'column': 'monto',
  'new_column': 'monto_median_1d',
  'params': {'window': '1d'}},
 {'process': 'max_window',
  'column': 'monto',
  'new_column': 'monto_max_1d',
  'params': {'wi

In [5]:
af = AggregationFeatures(df, group_column=ENTITY_COL, date_column=DATE_COL)
agg_features_df = af.apply_transformations(config_list)

print(f"Shape features nuevas: {agg_features_df.shape}")
print("\n% de nulos por feature (sin transacciones previas de esa entidad en la ventana):")
print((agg_features_df.isna().mean() * 100).round(1))
agg_features_df.head()


Shape features nuevas: (104981, 30)

% de nulos por feature (sin transacciones previas de esa entidad en la ventana):
monto_sum_1h        77.9
monto_mean_1h       77.9
monto_median_1h     77.9
monto_max_1h        77.9
monto_min_1h        77.9
monto_sum_1d        30.1
monto_mean_1d       30.1
monto_median_1d     30.1
monto_max_1d        30.1
monto_min_1d        30.1
monto_sum_3d        16.8
monto_mean_3d       16.8
monto_median_3d     16.8
monto_max_3d        16.8
monto_min_3d        16.8
monto_sum_7d        10.8
monto_mean_7d       10.8
monto_median_7d     10.8
monto_max_7d        10.8
monto_min_7d        10.8
monto_sum_10d        9.2
monto_mean_10d       9.2
monto_median_10d     9.2
monto_max_10d        9.2
monto_min_10d        9.2
monto_sum_15d        8.0
monto_mean_15d       8.0
monto_median_15d     8.0
monto_max_15d        8.0
monto_min_15d        8.0
dtype: float64


,monto_sum_1h,monto_mean_1h,monto_median_1h,monto_max_1h,monto_min_1h,monto_sum_1d,monto_mean_1d,monto_median_1d,monto_max_1d,monto_min_1d,monto_sum_3d,monto_mean_3d,monto_median_3d,monto_max_3d,monto_min_3d,monto_sum_7d,monto_mean_7d,monto_median_7d,monto_max_7d,monto_min_7d,monto_sum_10d,monto_mean_10d,monto_median_10d,monto_max_10d,monto_min_10d,monto_sum_15d,monto_mean_15d,monto_median_15d,monto_max_15d,monto_min_15d
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Nota sobre los nulos:** son esperados y crecientes hacia ventanas cortas — reflejan que la entidad no tuvo otra transacción previa dentro de esa ventana (p.ej. ventana de 1h: la mayoría de las entidades no transaccionan dos veces en la misma hora). No se imputan aquí; cada prueba estadística de la siguiente sección descarta nulos por columna, igual que en `01_statistical_analysis.ipynb`.

In [6]:
train_agg_features = pd.concat([df, agg_features_df], axis=1)
print(f"Shape final (originales + agregaciones): {train_agg_features.shape}")

train_agg_features.to_csv(OUTPUT_TRAIN_PATH, index=False)
print(f"Guardado: {OUTPUT_TRAIN_PATH}")


Shape final (originales + agregaciones): (104981, 49)
Guardado: /Users/dpiedrahita/proyectos/DS_pro/dataset/train_agg_features.csv


## Análisis estadístico: agregaciones vs. target

Mismo criterio de `01_statistical_analysis.ipynb`: todas estas 25 features son continuas (`sum`/`mean`/`median`/`max`/`min` de `monto`), así que se les corre la batería completa de pruebas continuas (`FeatureStatTests.compute(..., kind="continuous")`): Mann-Whitney U, KS, correlación punto-biserial, ROC-AUC (`auc_abs`), d de Cohen, y Wald sobre regresión logística univariada. Mismos umbrales de interpretación embebidos (AUC: <0.6 casi aleatorio, 0.6-0.7 débil, 0.7-0.8 moderada, ≥0.8 fuerte).

In [7]:
fts = FeatureStatTests(train_agg_features)

agg_cols = list(agg_features_df.columns)
agg_results = [fts.compute(col, TARGET_COL, kind="continuous") for col in agg_cols]

stats_fraud_agg = pd.DataFrame(agg_results).sort_values("auc_abs", ascending=False).reset_index(drop=True)
stats_fraud_agg.to_csv(AGG_STATS_PATH, index=False)
print(f"Guardado: {AGG_STATS_PATH}")
stats_fraud_agg


Guardado: /Users/dpiedrahita/proyectos/DS_pro/02_feature_engineer/agg_stats_outputs/stats_fraud_agg_features.csv


,variable,mannwhitney_stat,mannwhitney_p_value,ks_stat,ks_p_value,point_biserial_corr,point_biserial_p_value,auc,auc_abs,auc_interpretation,cohens_d,cohens_d_interpretation,logit_beta,logit_odds_ratio,logit_p_value,logit_separation_detected
0,monto_sum_15d,201384909.0,9.012921e-68,0.101714,2.413920e-44,0.063739,1.804324e-87,0.571937,0.571937,casi aleatorio,0.284302,pequeño,0.000008,1.000008,1.031009e-75,False
1,monto_sum_10d,197795354.5,2.731453e-62,0.099410,6.572912e-42,0.063135,9.318872e-85,0.569264,0.569264,casi aleatorio,0.281305,pequeño,0.000010,1.000010,7.475358e-74,False
2,monto_max_15d,203085451.5,2.594752e-61,0.092489,1.000244e-36,0.078050,2.572018e-130,0.568322,0.568322,casi aleatorio,0.348490,pequeño,0.000642,1.000642,2.581605e-118,False
3,monto_max_10d,198258073.0,1.522983e-60,0.092387,3.027619e-36,0.076545,8.627742e-124,0.568256,0.568256,casi aleatorio,0.341377,pequeño,0.000672,1.000672,4.892919e-112,False
4,monto_max_7d,192591183.0,3.494923e-59,0.093803,6.682780e-37,0.073183,2.313621e-111,0.567879,0.567879,casi aleatorio,0.325459,pequeño,0.000699,1.000699,4.116374e-101,False
5,monto_sum_7d,193016467.5,1.396337e-57,0.095414,3.624677e-38,0.062717,3.051412e-82,0.566925,0.566925,casi aleatorio,0.278717,pequeño,0.000014,1.000014,3.320313e-72,False
6,monto_max_3d,170681695.0,6.437126e-53,0.092890,2.999893e-34,0.072114,4.631749e-101,0.565885,0.565885,casi aleatorio,0.318541,pequeño,0.000795,1.000795,7.342408e-90,False
7,monto_sum_3d,172072213.0,1.415988e-47,0.089495,8.083487e-32,0.064524,3.014106e-81,0.562348,0.562348,casi aleatorio,0.284866,pequeño,0.000030,1.000030,1.179217e-71,False
8,monto_max_1d,127060098.5,2.974518e-35,0.088404,4.804632e-27,0.069641,1.480347e-79,0.557192,0.557192,casi aleatorio,0.302371,pequeño,0.000928,1.000929,1.208890e-69,False
9,monto_mean_15d,210207174.5,7.516306e-38,0.078837,8.631223e-27,0.055768,2.269220e-67,0.553185,0.553185,casi aleatorio,0.248631,pequeño,0.002416,1.002419,1.537541e-55,False


In [8]:
print("Top 10 features de agregación por auc_abs:")
display(stats_fraud_agg[["variable", "auc_abs", "auc_interpretation", "cohens_d", "cohens_d_interpretation"]].head(10))

print("\nDistribución de auc_interpretation entre las features de agregación:")
print(stats_fraud_agg["auc_interpretation"].value_counts())

print("\nComparación de referencia — auc_abs de monto (variable original) del análisis anterior:", 0.559277)


Top 10 features de agregación por auc_abs:


,variable,auc_abs,auc_interpretation,cohens_d,cohens_d_interpretation
0,monto_sum_15d,0.571937,casi aleatorio,0.284302,pequeño
1,monto_sum_10d,0.569264,casi aleatorio,0.281305,pequeño
2,monto_max_15d,0.568322,casi aleatorio,0.348490,pequeño
3,monto_max_10d,0.568256,casi aleatorio,0.341377,pequeño
4,monto_max_7d,0.567879,casi aleatorio,0.325459,pequeño
5,monto_sum_7d,0.566925,casi aleatorio,0.278717,pequeño
6,monto_max_3d,0.565885,casi aleatorio,0.318541,pequeño
7,monto_sum_3d,0.562348,casi aleatorio,0.284866,pequeño
8,monto_max_1d,0.557192,casi aleatorio,0.302371,pequeño
9,monto_mean_15d,0.553185,casi aleatorio,0.248631,pequeño



Distribución de auc_interpretation entre las features de agregación:
auc_interpretation
casi aleatorio    30
Name: count, dtype: int64

Comparación de referencia — auc_abs de monto (variable original) del análisis anterior: 0.559277


## Conclusiones

**Basado en los resultados reales de `stats_fraud_agg` (30 features evaluadas, incluida la ventana de 15d agregada después):**

- **Ninguna de las 30 features individualmente supera el umbral de "casi aleatorio"** (todas quedan por debajo de auc_abs=0.6). El tamaño de efecto (Cohen's d) es "pequeño" o "despreciable" en todos los casos.
- **La ventana de 15d no cambia la conclusión, pero sí corre el líder**: `monto_sum_15d` (auc_abs=0.572) pasa a ser la mejor de las 30, superando a `monto_sum_10d` (0.569) y `monto_max_15d` (0.568). Confirma el mismo patrón visto antes: mientras más larga la ventana, más historial acumula la entidad y más estable el agregado — la mejora de 10d a 15d es marginal (+0.003), señal de que el aporte ya se está aplanando (rendimientos decrecientes al alargar la ventana).
- Se mantiene el mismo patrón por tipo de agregación: **`sum`/`max` en ventanas largas > ventanas cortas (1h) > `median`/`min`**, que rondan el azar puro (`monto_min_3d`, auc_abs=0.5005; `monto_min_15d`, 0.520).
- **Comparadas contra `monto` original (auc_abs=0.559)**, ahora son 5 las que lo superan (antes 4): `monto_sum_15d` (0.572), `monto_sum_10d` (0.569), `monto_max_15d` (0.568), `monto_max_10d` (0.568) y `monto_max_7d`/`monto_sum_7d` (0.568/0.567) — todas por un margen pequeño (~0.01-0.013).

**Conclusión (sin cambios de fondo tras agregar 15d):** estas features de agregación **siguen sin aportar poder predictivo individual relevante** — `j` como proxy de entidad genera historiales muy cortos (~12-13 transacciones en 44 días) y demasiado dispersos en el tiempo para que "cuánto gastó antes" separe bien fraude de no-fraude por sí solo, incluso extendiendo la ventana a 15 días. No se recomienda incorporar las 30 tal cual: si se quiere probarlas en un modelo multivariado, tiene más sentido quedarse con 2-3 candidatas (`monto_sum_15d`, `monto_max_10d`) en vez de agregar 30 columnas altamente correlacionadas entre sí por un beneficio marginal. Dado que alargar la ventana de 10d a 15d ya no mejora casi nada, no parece necesario seguir probando ventanas aún más largas (20d, 30d) — el techo de esta familia de features parece estar cerca de auc_abs≈0.57.

## Consolidación

Aunque estas 30 features de agregación no mostraron poder predictivo individual relevante, se consolidan igual con el resto del feature engineering (`dataset/train_feat_engineer.csv`, 80 columnas) en un único archivo `train_all_features.csv`, para tener todo el trabajo de las secciones 00-02 en un solo dataset y decidir su uso final en la etapa de modelado (selección de variables / regularización), no aquí.

El join se hace por `k` (identificador único de transacción, presente en ambos archivos), agregando solo las 30 columnas de agregación nuevas — las 19 columnas originales ya están en `train_feat_engineer.csv` y no se duplican.

In [9]:
TRAIN_FEAT_ENGINEER_PATH = DATASET_DIR / "train_feat_engineer.csv"
OUTPUT_ALL_FEATURES_PATH = DATASET_DIR / "train_all_features.csv"

train_feat_engineer = pd.read_csv(TRAIN_FEAT_ENGINEER_PATH, low_memory=False)
print(f"train_feat_engineer: {train_feat_engineer.shape}")

agg_only_cols = ["k"] + list(agg_features_df.columns)
train_all_features = train_feat_engineer.merge(
    train_agg_features[agg_only_cols], on="k", how="left", validate="one_to_one"
)

print(f"train_all_features: {train_all_features.shape}")
assert len(train_all_features) == len(train_feat_engineer), "el join no debería cambiar el número de filas"

train_all_features.to_csv(OUTPUT_ALL_FEATURES_PATH, index=False)
print(f"Guardado: {OUTPUT_ALL_FEATURES_PATH}")
train_all_features.head()


train_feat_engineer: (104981, 80)
train_all_features: (104981, 110)
Guardado: /Users/dpiedrahita/proyectos/DS_pro/dataset/train_all_features.csv


,a,b,c,d,e,f,g,h,j,k,l,m,n,o,p,fecha,monto,score,fraude,a_freq_enc,a_grouped,b_missing,b_is_zero,b_log,b_top1pct,b_bin,c_missing,c_log,c_top1pct,c_bin,d_missing,d_is_zero,d_log,d_top1pct,d_bin,e_is_zero,e_log,e_top1pct,e_bin,f_missing,f_is_zero,f_log,f_top1pct,f_bin,g_missing,g_freq_enc,g_grouped,h_is_zero,h_log,h_top1pct,h_bin,j_freq_enc,j_grouped,j_target_enc,l_missing,l_is_zero,l_log,l_top1pct,l_bin,m_missing,m_is_zero,m_log,m_top1pct,m_bin,o_freq_enc,o_target_enc,fecha_hour,fecha_weekday,fecha_is_weekend,fecha_hour_sin,fecha_hour_cos,monto_log,monto_top1pct,monto_bin,score_is_zero,score_log,score_top1pct,score_bin,pca_dm_1,ratio_monto_l,monto_sum_1h,monto_mean_1h,monto_median_1h,monto_max_1h,monto_min_1h,monto_sum_1d,monto_mean_1d,monto_median_1d,monto_max_1d,monto_min_1d,monto_sum_3d,monto_mean_3d,monto_median_3d,monto_max_3d,monto_min_3d,monto_sum_7d,monto_mean_7d,monto_median_7d,monto_max_7d,monto_min_7d,monto_sum_10d,monto_mean_10d,monto_median_10d,monto_max_10d,monto_min_10d,monto_sum_15d,monto_mean_15d,monto_median_15d,monto_max_15d,monto_min_15d
0,4,0.7388,6314.50,14.0,0.139279,24.0,BR,7,cat_381751d,0.937548,2361.0,442.0,1,NaN,Y,2020-03-08 00:02:15,22.18,25,0,0.850821,4,0,0,0.553195,0,2,0,8.750762,0,1,0,0,2.708050,0,2,0,0.130395,0,0,0,0,3.218876,0,3,0,0.755489,BR,0,2.079442,0,1,0.004563,__OTHER__,0.042624,0,0,7.767264,0,2,0,0,6.093570,0,3,0.724388,0.051838,0,6,1,0.0,1.0,3.143290,0,2,0,3.258097,0,1,142.485292,0.009394,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4,0.7548,21171.09,20.0,0.514815,7.0,BR,2,cat_a024847,0.791998,2324.0,73.0,1,NaN,N,2020-03-08 00:04:25,6.00,7,0,0.850821,4,0,0,0.562355,0,2,0,9.960439,0,1,0,0,3.044522,0,2,0,0.415293,0,2,0,0,2.079442,0,2,0,0.755489,BR,0,1.098612,0,0,0.000133,__OTHER__,0.023563,0,0,7.751475,0,2,0,0,4.304065,0,1,0.724388,0.051838,0,6,1,0.0,1.0,1.945910,0,0,0,2.079442,0,0,-226.040600,0.002582,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4,0.9026,4012.83,50.0,0.274167,1.0,BR,3,cat_1d61c62,0.688592,235.0,232.0,1,N,Y,2020-03-08 00:08:23,26.67,91,1,0.850821,4,0,0,0.643221,0,4,0,8.297501,0,0,0,0,3.931826,1,3,0,0.242292,0,1,0,0,0.693147,0,0,0,0.755489,BR,0,1.386294,0,1,0.002648,__OTHER__,0.044569,0,0,5.463832,0,0,0,0,5.451038,0,2,0.113421,0.230557,0,6,1,0.0,1.0,3.320349,0,2,0,4.521789,0,4,-66.040159,0.113489,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,0.8285,99612.95,1.0,0.000000,4.0,BR,28,cat_01a1725,0.654161,658.0,0.0,1,N,N,2020-03-08 00:08:39,40.01,91,1,0.850821,4,0,0,0.603496,0,4,0,11.509057,0,3,0,0,0.693147,0,0,1,0.000000,0,0,0,0,1.609438,0,1,0,0.755489,BR,0,3.367296,0,4,0.000438,__OTHER__,0.119965,0,0,6.490724,0,0,0,1,0.000000,0,0,0.113421,0.228374,0,6,1,0.0,1.0,3.713816,0,3,0,4.521789,0,4,-299.693151,0.060805,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,0.5992,53526.36,2.0,0.000000,264.0,AR,34,cat_f1e7464,0.532994,2400.0,10.0,1,NaN,N,2020-03-08 00:09:01,6.25,93,0,0.101837,2,0,0,0.469504,0,0,0,10.887948,0,2,0,0,1.098612,0,0,1,0.000000,0,0,0,0,5.579730,0,4,0,0.204132,AR,0,3.555348,0,4,0.000076,__OTHER__,0.032399,0,0,7.783641,0,2,0,0,2.397895,0,0,0.724388,0.051838,0,6,1,0.0,1.0,1.981001,0,0,0,4.543295,0,4,-289.663017,0.002604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
